# KRSB на 20 Newsgroups

Пример Keyword Random Subspace Bagging: три экстрактора ключевых слов (YAKE, RAKE, TopicRank) строят представления документов, затем ансамбль логистических голов голосует по случайным подпространствам ключевых фраз.

Датасет скачивается автоматически через `sklearn.datasets.fetch_20newsgroups`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "examples":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from KRSB import KRSB, RakeExtractor, TfidfEncoder, TopicRankExtractor, YakeExtractor
from KRSB.bank import KeywordBank

In [ ]:
CATEGORIES = ["sci.space", "sci.med", "rec.autos", "talk.politics.misc"]
SAMPLES_PER_CLASS = 80
RANDOM_STATE = 42

data = fetch_20newsgroups(
    subset="all",
    categories=CATEGORIES,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_STATE,
)

texts, labels = [], []
counts = {i: 0 for i in range(len(CATEGORIES))}
for text, label in zip(data.data, data.target):
    if counts[label] >= SAMPLES_PER_CLASS:
        continue
    cleaned = " ".join(text.split())
    if len(cleaned) < 80:
        continue
    texts.append(cleaned)
    labels.append(label)
    counts[label] += 1
    if all(v >= SAMPLES_PER_CLASS for v in counts.values()):
        break

x_train, x_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=RANDOM_STATE, stratify=labels
)
print(f"train={len(x_train)} test={len(x_test)}")
print("classes:", list(data.target_names))

Извлекаем ключевые фразы тремя методами. `KeywordBank` — общая таблица представлений, которую потом можно передать в любой ансамбль.

In [ ]:
extractors = [
    YakeExtractor(ngram=3),
    RakeExtractor(),
    TopicRankExtractor(),
]

train_bank = KeywordBank.from_extractors(x_train, extractors, top_n=15)
test_bank = KeywordBank.from_extractors(x_test, extractors, top_n=15)

print("YAKE:", train_bank.row(0)["yake"][:5])
print("RAKE:", train_bank.row(0)["rake"][:5])
print("TopicRank:", train_bank.row(0)["topicrank"][:5])

Каждая голова KRSB берёт bootstrap документов и случайное подпространство методов, кодирует получившийся keyword-текст и учит логистическую регрессию. Предсказание — среднее вероятностей (soft voting).

В этом примере энкодер — TF-IDF, чтобы всё считалось на CPU без SciBERT. В оригинальных ноутбуках на его месте `BertEncoder("allenai/scibert_scivocab_uncased")` или дообученный encoder из `KRSB.finetune`.

In [ ]:
model = KRSB(
    encoder=TfidfEncoder(),
    n_estimators=10,
    methods_per_model=2,
    total_k=20,
    seed=RANDOM_STATE,
)
model.fit(train_bank, y_train)
pred = model.predict(test_bank)
print(classification_report(y_test, pred, target_names=data.target_names, digits=3))